# Dense, Sparse 데이터 저장 & 앙상블 검색기 테스트

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# 현재 노트북(src/notebooks) 위치 기준으로 상위 프로젝트 루트(08_RAG2)를 sys.path에 추가
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.hybrid_retriever import build_and_save_hybrid_store, load_hybrid_retriever

# 1. pdf -> parsing 한 md 으로 변환 (되어있어야 함) -> parsed_data/ 폴더 안에 있음

# 2. 마크다운을 split -> 저장하는 함수
build_and_save_hybrid_store(
    markdown_path='parsed_data/output.md',
    source_name="NIA_2026_trends.pdf"
)

In [ ]:
# 3. 검색기를 활용
retriever = load_hybrid_retriever(dense_k=5, sparse_k=5)


In [ ]:
docs = retriever.invoke('글로벌 AI 반도체 시장 규모 2026년 전망')

In [ ]:
from pprint import pprint
for doc in docs: 
    pprint(doc.page_content)

In [ ]:
from src.hybrid_retriever import build_and_save_hybrid_store
# 2. 마크다운을 split -> 저장하는 함수
build_and_save_hybrid_store(
    markdown_path='parsed_data/output.md',
    source_name="NIA_2026_trends.pdf"
)


## 우리가 그래프에서 쓸 코드

In [3]:

from src.rerank_retriever import build_reranked_retriever

final_powerful_retriever = build_reranked_retriever(top_n=5)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [4]:
final_powerful_retriever.invoke('피지컬 AI의 미래와 인간의 존재 이유')

[Document(metadata={'source': 'NIA_2026_trends.pdf', 'type': 'text', 'h1': '트렌드 3 AI가 현실 세계로, 산업 현장에서 시작되는 피지컬 AI 혁신', 'h2': '트렌드 배경', 'h3': '인구구조 변화와 산업적 수요'}, page_content='### 인구구조 변화와 산업적 수요  \n* 글로벌 고령화와 노동력 부족이라는 구조적 문제가 심화되면서, 피지컬 AI가 전 산업 분야에서 생산성과 효율성을 획기적으로 개선하고 인간의 업무를 보조·대체하는 핵심 수단으로 주목'),
 Document(id='24326f5c-4514-437d-a72e-2cb8cc6ee670', metadata={'h1': '트렌드 3 AI가 현실 세계로, 산업 현장에서 시작되는 피지컬 AI 혁신', 'source': 'NIA_2026_trends.pdf', 'type': 'text', 'h2': '트렌드 배경'}, page_content='## 트렌드 배경  \n> **피지컬 AI는 단순한 자동화나 로봇을 넘어, AI 기술의 진화가 디지털 공간을 벗어나 현실 세계의 문제 해결에 직접 적용되는 새로운 패러다임**'),
 Document(metadata={'source': 'NIA_2026_trends.pdf', 'type': 'text', 'h1': '2026년 AI·디지털 트렌드', 'h2': 'AI·디지털 트렌드 핵심 내용 및 전망'}, page_content='# 2026년 AI·디지털 트렌드  \n## AI·디지털 트렌드 핵심 내용 및 전망  \nicon: AI infrastructure  \nAI의 새 격전지, AI 인프라 패권 경쟁 심화  \n* 글로벌 주요국은 반도체를 전략 산업으로 육성하며 AI 반도체 시장 다각화 및 개발 경쟁 가속화  \n* AI 컴퓨팅 자원 동맹 및 블록화 현상이 심화될 전망  \nicon: AI agent  \n스스로 일하는 AI 에이전트, 협업과 자동화로 재편되는 미래  \n* 인간의 지

In [ ]:
# RAG 툴 만들고 싶다면?
from langchain.tools import tool

@tool
def rag_tool(query: str):
    retriever = build_reranked_retriever(top_n=5)
    result = retriever.invoke(query)
    # 결과 1개의 거대한 str 으로 합치는 코드
    return result